BLOCO 1 — Carregar modelo salvo

In [ ]:
import joblib
import pandas as pd

MODEL_PATH = "../models/grupo/grupo.pkl"
DATA_PATH = "../data/processed/comentarios_processados_full.csv"

artefato = joblib.load(MODEL_PATH)
model = artefato["model"]
vectorizer = artefato["vectorizer"]

df = pd.read_csv(DATA_PATH)
df.head()

BLOCO 2 — Preparar dados igual ao treino

In [20]:
df = df.dropna(subset=["grupo", "comentario", 'fase'])

df["comentario"] = df["comentario"].astype(str).str.strip()
df = df[df["comentario"].str.len() >= 5]
df = df[df["fase"] == "p1"]

df["grupo"] = (
    df["grupo"]
    .str.upper()
    .str.replace(" ", "_")
)

BLOCO 3 — Fazer previsão no dataset inteiro

In [25]:
X = df["comentario"]
y_true = df["grupo"]

X_vec = vectorizer.transform(X)
y_pred = model.predict(X_vec)

df["y_true"] = y_true
df["y_pred"] = y_pred

In [26]:
def aplicar_regras(texto, predicao_modelo):
    texto_lower = texto.lower()

    palavras_admin = [
        "maqueiro", "porteiro", "portaria",
        "recepcao", "recepcionista",
        "totem", "enfermeiro", "enfermeira"
    ]

    count = sum(p in texto_lower for p in palavras_admin)

    if count >= 2:
        return "ADMINISTRATIVO"

    if "totem" in texto_lower and any(
        p in texto_lower for p in ["ninguém", "auxilio", "orientacao", "ajuda"]
    ):
        return "ADMINISTRATIVO"

    return predicao_modelo

In [27]:
y_pred_modelo = model.predict(X_vec)

y_pred_final = [
    aplicar_regras(texto, pred)
    for texto, pred in zip(df["comentario"], y_pred_modelo)
]

BLOCO 4 — Extrair erros

In [ ]:
df_erros = df[df["y_true"] != df["y_pred"]]
df_erros.head(20)

BLOCO 5 — Separar tipos de erro

In [ ]:
erros_administrativo = df[
    (df["y_true"] == "ADMINISTRATIVO") &
    (df["y_pred"] == "RECEPÇAO")
]

erros_administrativo[["comentario"]].head(50)

In [ ]:
erros_recepcao = df[
    (df["y_true"] == "RECEPÇAO") &
    (df["y_pred"] == "ADMINISTRATIVO")
]

erros_recepcao[["comentario"]].head(20)

In [29]:
df_erros = df[df["y_true"] != df["y_pred"]].copy()

df_erros["tipo_erro"] = df_erros["y_true"] + " → " + df_erros["y_pred"]

df_erros.to_excel("erros_modelo_grupo.xlsx", index=False)